# 第4章 第一个程序 + 性能分析

> 前面几章我们铺开了环境验证和 GPU 体系结构两张地图。现在，地图已经在你手上了——终于到了**写一点代码、量一组数字**的时候。本章会做三件事：跑通第一个手写的 HIP kernel（vector add）、建立一个可复用的基准测试，再读懂量出来的这组数字——用算术强度和带宽利用率，建立「这个算子离硬件极限有多远」的直觉。vector add 会贯穿整个 Part 1 profiling 篇，所以这章是后面所有实验的起点。

**本章目标**：跑通 vector add kernel，建立基准测试，理解算术强度和有效带宽。

**前置条件**：已完成第1-3章，理解 GPU 体系结构和内存层级。

**平台**：原生 Ubuntu 24.04（推荐）或 WSL2，gfx1201 为叙述基线。

本章对应代码在：

```text
code/part0-intro/
├── pyproject.toml
├── uv.lock
├── activate-rocm.sh
└── chapter4/
    ├── vector_add.hip
    └── benchmark_vector_add.py
```

## 4.1 从已经验证的环境开始

在写第一行 GPU 代码之前，有一件事必须确认：环境是通的。好在这件事第 1 章已经帮你做完了——三道环境验证门（`rocminfo` 能看到 GPU、PyTorch ROCm 能跑 GPU tensor、最小 HIP 程序能编译运行）都已经通过。如果你还没做，请先回去跑完那三道门。

进入本篇环境：

```bash
cd code/part0-intro
uv sync
source ./activate-rocm.sh
```

下面先定位仓库根目录并检测当前 GPU 架构，确认环境可用。

## 在云端运行本章

本教程以 RX 9070 XT（`gfx1201` / RDNA4）为讲解和参考环境。云端结果用于验证代码并观察同一平台内的变化，不应与参考数据或其他 GPU 的绝对性能直接比较。

云平台已预装 ROCm、PyTorch 和基础编译工具，可跳过本地的 `uv sync` 与环境激活步骤；本地读者仍按原步骤准备环境。本章所需的额外依赖会在章节内单独提示。

请根据当前 `rocminfo` 输出选择编译架构；未识别时先检查环境。


In [ ]:
# 检测当前 GPU 架构
import os
import re
import subprocess
import sys

ARCH_DETECT_TIMEOUT_S = 10
COMPILE_TIMEOUT_S = 120
SMOKE_TIMEOUT_S = 120
FULL_TIMEOUT_S = 300
SUPPORTED_ARCHES = {"gfx1100", "gfx1151", "gfx1201"}
GPU_AGENT_BLOCK_RE = re.compile(
    r"(?ms)^\s*Agent\s+\d+\s*$.*?(?=^\s*Agent\s+\d+\s*$|\Z)"
)
GPU_AGENT_NAME_RE = re.compile(r"(?m)^\s*Name:\s*(gfx[0-9a-z]+)\s*$")
WAVEFRONT_SIZE_RE = re.compile(r"(?m)^\s*Wavefront Size:\s*(\d+)\s*$")


def _gpu_agent_details(rocminfo_stdout):
    """Return gfx names and optional wavefront sizes from GPU Agent blocks only."""
    candidates = {}
    for block in GPU_AGENT_BLOCK_RE.findall(rocminfo_stdout):
        if not re.search(r"(?m)^\s*Device Type:\s*GPU\s*$", block):
            continue
        wavefront_match = WAVEFRONT_SIZE_RE.search(block)
        wavefront_size = int(wavefront_match.group(1)) if wavefront_match else None
        for name in GPU_AGENT_NAME_RE.findall(block):
            candidates[name] = wavefront_size
    return candidates


def detect_architecture():
    override = os.environ.get("HELLO_GPU_ARCH", "").strip()
    if override:
        if override not in SUPPORTED_ARCHES:
            raise RuntimeError(
                f"HELLO_GPU_ARCH 仅支持 {sorted(SUPPORTED_ARCHES)}；实际值={override!r}"
            )
        return override, "override", None

    try:
        result = subprocess.run(
            ["rocminfo"],
            capture_output=True,
            text=True,
            timeout=ARCH_DETECT_TIMEOUT_S,
            check=False,
        )
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(
            f"rocminfo 超时（timeout={ARCH_DETECT_TIMEOUT_S}s）\n"
            f"stdout:\n{exc.stdout or ''}\n"
            f"stderr:\n{exc.stderr or ''}"
        ) from exc
    except FileNotFoundError as exc:
        raise RuntimeError(
            f"rocminfo 无法启动（returncode=NOT_STARTED）\nstderr:\n{exc}"
        ) from exc

    if result.returncode != 0:
        raise RuntimeError(
            f"rocminfo 失败（returncode={result.returncode}）\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}"
        )

    candidates = _gpu_agent_details(result.stdout)
    if len(candidates) != 1:
        raise RuntimeError(
            "rocminfo 必须恰好报告一个 GPU Agent Name: gfx...；"
            f"实际候选={sorted(candidates) or 'none'}"
        )
    arch, wavefront_size = next(iter(candidates.items()))
    return arch, "rocminfo", wavefront_size


def run_checked(command, *, cwd, label, timeout=COMPILE_TIMEOUT_S):
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            cwd=cwd,
            timeout=timeout,
            check=False,
        )
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(
            f"{label} 超时（timeout={timeout}s；returncode=TIMEOUT）\n"
            f"stdout:\n{exc.stdout or ''}\n"
            f"stderr:\n{exc.stderr or ''}"
        ) from exc
    except FileNotFoundError as exc:
        raise RuntimeError(
            f"{label} 无法启动（returncode=NOT_STARTED）\nstderr:\n{exc}"
        ) from exc
    if result.returncode != 0:
        raise RuntimeError(
            f"{label} 失败（returncode={result.returncode}）\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}"
        )
    return result


arch, arch_source, wavefront_size = detect_architecture()
if arch not in SUPPORTED_ARCHES:
    raise RuntimeError(
        f"当前架构 {arch} 暂无本章对应 vector-add 说明；支持集合={sorted(SUPPORTED_ARCHES)}"
    )
print(f"arch={arch}; arch_source={arch_source}")
if arch_source == "override":
    print("wavefront_size=unknown（override 不证明硬件；以运行结果为准）")
elif wavefront_size is None:
    print("wavefront_size=unknown（GPU Agent 未报告；以运行结果为准）")
else:
    print(f"wavefront_size={wavefront_size}")


In [ ]:
import pathlib
import subprocess

def find_repo_root():
    current = pathlib.Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").is_dir()
            and (candidate / "docs").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise FileNotFoundError("无法定位仓库根目录")

REPO_ROOT = find_repo_root()
print(f"仓库根目录: {REPO_ROOT}")

## 4.2 跑通 vector add

环境就绪，开始写代码。一个最小但完整的 HIP 程序通常包含六步：Host 准备数据、Device 分配显存、Host 到 Device 拷贝、启动 kernel、Device 到 Host 拷回、检查结果。这六步构成了所有 GPU 程序的骨架，后面的 Reduction、Softmax、Matmul 无论多复杂，骨架都是这六步。

*（图示：最小 HIP Vector Add 程序的数据路径）*

> **数据路径**：Host 输入 → `hipMalloc` → `hipMemcpy H2D` → **Kernel Launch** → `hipMemcpy D2H` → 校验结果

### Kernel 代码

完整的 `vector_add.hip` 文件在第 1 章 1.6 节已经作为环境验证出现过。**真正值得重点看的是中间那段 kernel**，它才是跑在 GPU 上的代码：

```cpp
__global__ void vector_add(const float* a, const float* b, float* c, int n) {
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx < n) {
    c[idx] = a[idx] + b[idx];
  }
}
```

这段 kernel 的映射关系很直接：一个 GPU 线程负责一个元素。`blockIdx.x * blockDim.x + threadIdx.x` 计算出当前线程负责的全局下标，`if (idx < n)` 用来处理最后一个 block 可能越界的情况。

### 新语法速查

- `__global__`：告诉编译器"这是一个 GPU 函数，由 CPU 调用、在 GPU 上运行"；
- `<<<blocks, threads>>>`：HIP / CUDA 特有的 kernel 启动语法，`blocks` 是要启动多少组，`threads` 是每组多少个线程；
- `blockIdx.x` / `threadIdx.x`：每个 GPU 线程拿到的"工号"，用它来算自己负责数组里的哪个位置。

换成大白话：`vector_add<<<blocks, threads>>>(...)` 就是在 GPU 上同时叫起 `blocks × threads` 个工人，每个工人执行一次 `vector_add` 函数。

*（图示：Grid → Block → Thread 的层级，以及 blockIdx/threadIdx 如何算出全局下标）*

### 编译并运行

编译选项说明：
- 源文件：`code/part0-intro/chapter4/vector_add.hip`
- 编译选项：`-O2`
- 目标架构：根据实际 GPU 自动选择（gfx1100 / gfx1151 / gfx1201）

预期输出：设备名称、向量大小、blocks 和 threads_per_block、max_error 为 0、status: PASS。

In [ ]:
chapter4_dir = REPO_ROOT / "code/part0-intro/chapter4"
vector_add_hip = chapter4_dir / "vector_add.hip"
vector_add_bin = chapter4_dir / "vector_add"

compile_result = run_checked(
    ["hipcc", f"--offload-arch={arch}", "-O2",
     str(vector_add_hip), "-o", str(vector_add_bin)],
    cwd=chapter4_dir,
    label="vector_add.hip 编译",
    timeout=COMPILE_TIMEOUT_S,
)
print("编译成功")

# 先做一次小规模检查；run_checked 会在失败或超时时显示 returncode、stdout、stderr 并停止。
run_result = run_checked(
    [str(vector_add_bin)],
    cwd=chapter4_dir,
    label="Vector Add 小规模检查",
    timeout=SMOKE_TIMEOUT_S,
)
print("\n运行结果:")
print(run_result.stdout)

if "status: PASS" not in run_result.stdout:
    raise RuntimeError(
        "Vector Add 返回非 PASS，停止以避免继续消费旧结果；"
        f"stdout:\n{run_result.stdout}"
    )
print("\n✓ Vector Add 验证通过")

看到 `status: PASS` 后，你已经跑通了第一段真正由自己编译的 GPU kernel——欢迎正式进入 GPU 编程的世界。

## 4.3 建立基准测试

这一节加入一个很小的 benchmark。它不是为了证明 Vector Add 有多快，而是为了提前建立后续章节会反复使用的习惯：固定输入规模、先 warmup、重复运行多次、记录 mean / median / min。

记住三件事：

1. 正式计时前先 warmup 5 次，让 GPU 进入比较稳定的状态；
2. 正式跑 30 次，每次用 `torch.cuda.Event` 量 GPU 真正执行完成的时间；
3. 最后用最小值估算带宽，因为最小值更像"没被外部干扰"的那次。

*（图示：准确 benchmark 前先 warmup、多次 repeat，并在同步后计时）*

### GPU 计时的关键

GPU 计时最关键的是使用 event 并在每轮后同步：

```python
start.record()
c = a + b
end.record()
torch.cuda.synchronize()
times.append(start.elapsed_time(end))
```

否则 CPU 端可能只是把任务提交出去，计到的不是 GPU 真正执行完成的时间。

### 带宽估算模型

这里的带宽估算使用的是 Vector Add 的最简单数据量模型。一次 Vector Add 要读 `a`、读 `b`、写 `c`，一共经过 3 个数组；每个元素是 `float32`，也就是 4 字节。所以一次完整 Vector Add 搬动的数据量是：

```text
bytes_moved = vector_size × 3 × 4
```

下面运行 benchmark（向量大小 16M 元素 = $2^{24}$，warmup 5 次，repeat 30 次）：

In [ ]:
benchmark_script = chapter4_dir / "benchmark_vector_add.py"

run_result = run_checked(
    [sys.executable, str(benchmark_script),
     "--size", "16777216",  # 2^24
     "--warmup", "5",
     "--repeat", "30"],
    cwd=chapter4_dir,
    label="Vector Add 完整 benchmark",
    timeout=FULL_TIMEOUT_S,
)
print("Benchmark 结果:")
print(run_result.stdout)


def parse_benchmark_output(stdout):
    """Parse the stable key:value lines emitted by benchmark_vector_add.py."""
    integer_fields = {"vector_size", "warmup", "repeat"}
    float_fields = {
        "cpu_mean_ms", "cpu_median_ms", "cpu_min_ms",
        "cpu_bandwidth_gb_s_by_min", "gpu_mean_ms", "gpu_median_ms",
        "gpu_min_ms", "gpu_bandwidth_gb_s_by_min",
    }
    parsed = {}
    for line in stdout.splitlines():
        key, separator, value = line.partition(":")
        key = key.strip()
        if not separator:
            continue
        value = value.strip()
        if key in integer_fields:
            parsed[key] = int(value)
        elif key in float_fields:
            parsed[key] = float(value)
        elif key in {"torch", "cuda_available", "device_name", "status"}:
            parsed[key] = value

    required = {
        "device_name", "vector_size", "warmup", "repeat",
        "cpu_mean_ms", "cpu_median_ms", "cpu_min_ms",
        "cpu_bandwidth_gb_s_by_min", "gpu_mean_ms", "gpu_median_ms",
        "gpu_min_ms", "gpu_bandwidth_gb_s_by_min", "status",
    }
    missing = sorted(required - parsed.keys())
    if missing:
        raise RuntimeError(f"benchmark 输出缺少字段: {missing}")
    if parsed["status"] != "PASS":
        raise RuntimeError(f"benchmark 未通过: status={parsed['status']}")
    return parsed


benchmark_result = parse_benchmark_output(run_result.stdout)
print("\n已解析当前平台结果；4.4 节将直接使用本次输出，不使用固定平台数值。")
print("\n关键指标说明:")
print("- warmup: 预热次数，让 GPU 进入稳定状态")
print("- repeat: 重复测量次数，用于统计分析")
print("- min_ms: 最小执行时间，更接近无干扰的真实性能")
print("- bandwidth_gb_s: 有效带宽 = (3 × N × 4 bytes) / time")


## 4.4 性能分析：CPU vs GPU 与带宽利用率

跑通和计时都做好了，现在来读懂本次运行实际产生的数字。下方代码块直接读取 4.3 的 `benchmark_result`；换一台 GPU 重新运行时，设备名、时间、加速比和有效带宽会随本次输出变化。

### 4.4.1 当前平台 CPU vs GPU

先完整打印 mean / median / min，而不是只挑一个固定参考值。CPU 与 GPU 的带宽均按 `3 × N × 4 Byte` 和各自 min 时间计算。


In [ ]:
def print_current_platform_result(result):
    device_name = result["device_name"]
    print("当前平台实测")
    print(f"- arch: {arch}")
    print(f"- device_name: {device_name}")
    print(f"- vector_size: {result['vector_size']:,}")
    print(f"- warmup / repeat: {result['warmup']} / {result['repeat']}")

    for statistic in ("mean", "median", "min"):
        cpu_ms = result[f"cpu_{statistic}_ms"]
        gpu_ms = result[f"gpu_{statistic}_ms"]
        speedup = cpu_ms / gpu_ms
        print(
            f"- {statistic}: CPU={cpu_ms:.6f} ms, "
            f"GPU={gpu_ms:.6f} ms, CPU/GPU={speedup:.2f}x"
        )

    print(
        "- effective_bandwidth_by_min: "
        f"CPU={result['cpu_bandwidth_gb_s_by_min']:.3f} GB/s, "
        f"GPU={result['gpu_bandwidth_gb_s_by_min']:.3f} GB/s"
    )


print_current_platform_result(benchmark_result)


#### RX 9070 XT 参考解释

下面只保留原参考平台的详细解释，不把这些固定值当作当前云端输出：

| 指标 | CPU（单核） | GPU（仅 kernel） |
| ---- | ----: | ----: |
| 耗时（min） | ~7.36 ms | ~0.345 ms |
| 有效带宽（按 min 估算） | ~27.4 GB/s | ~583 GB/s |

该参考记录中 GPU 约快 **21 倍**。这个数字只描述当时的 RX 9070 XT + ROCm 7.13 记录；当前平台应以上方动态输出为准，不做跨 GPU 绝对排名。

还要注意当前教学脚本的 CPU 路径在计时区间内包含 `sum().item()`，GPU event 则只覆盖加法 kernel。因此这里的 CPU/GPU 比值适合帮助观察当前脚本行为，不是严格同口径的硬件峰值对比。真正公平的端到端比较还需要统一计时边界，并明确是否包含 Host↔Device 搬运。

GPU 的优势通常在数据已经留在显存、并且后续有多步计算可以复用它时最明显。对 vector add 这类简单操作，如果把 Host↔Device 搬运也算进去，整体差距会明显缩小。

### 4.4.2 算术强度：vector add 是内存密集型

为什么 vector add 这么快、却又「没什么计算量」？看它的**算术强度**（Arithmetic Intensity，记作 $AI$）——每搬 1 Byte 数据做多少次计算：

$$
AI = \frac{\text{FLOPs}}{\text{Bytes}}
$$

对 vector add 的每个 float32 元素：读 `a[i]`（4 Byte）、读 `b[i]`（4 Byte）、写 `c[i]`（4 Byte），共 12 Byte；只做 1 次加法，即 1 FLOP。代入得：

$$
AI = \frac{1\ \text{FLOP}}{12\ \text{Byte}} \approx 0.083\ \text{FLOP/Byte}
$$

0.083 极低——每搬 12 字节才做 1 次加法。这意味着 vector add 是**典型的内存密集型（memory-bound）算子**：它的快慢几乎完全取决于显存带宽，而不是算力。把算力堆得再高，对 vector add 也没用，因为瓶颈在搬数据。

下面用代码验证这个计算：


In [ ]:
# 算术强度和带宽分析
N = benchmark_result["vector_size"]  # 使用本次 benchmark 的实际输入规模
bytes_per_element = 4  # FP32

# Vector Add: c[i] = a[i] + b[i]
# 读取: a[i], b[i] (2 reads)
# 写入: c[i] (1 write)
# 计算: 1 次加法

reads = 2 * N * bytes_per_element
writes = 1 * N * bytes_per_element
total_bytes = reads + writes
flops = N  # 1 次加法 per 元素

arithmetic_intensity = flops / total_bytes

print("算术强度分析:")
print(f"- 总元素数: {N:,}")
print(f"- 读取字节: {reads:,} bytes ({reads / 1e9:.3f} GB)")
print(f"- 写入字节: {writes:,} bytes ({writes / 1e9:.3f} GB)")
print(f"- 总字节数: {total_bytes:,} bytes ({total_bytes / 1e9:.3f} GB)")
print(f"- 浮点运算: {flops:,} FLOPs")
print(f"- 算术强度: {arithmetic_intensity:.6f} FLOP/Byte")

print("\n带宽概念:")
print("- 逻辑带宽: 基于算法字节数（3 × N × 4）计算的带宽")
print("- 物理带宽: 实际在 GDDR6 上传输的字节数（需硬件计数器）")
print("- 有效带宽 (effective bandwidth): 逻辑字节数 / 实测时间")
print("- 理论峰值: 必须从当前平台官方规格或项目实测基线中注明来源后填写")

print("\n解释:")
print("- Vector Add 是典型的访存密集型算子（arithmetic intensity 很低）")
print("- 性能瓶颈在内存带宽，而非计算能力")
print("- 优化重点是提高内存访问效率（合并访存、缓存利用）")

### 4.4.3 带宽利用率：离硬件极限有多远

先由本次 kernel 时间和搬运字节数动态计算**有效带宽** $BW_{effective}$（vector add 读 2 个、写 1 个，共 3 个数组、每元素 4 Byte）：

$$
BW_{effective} = \frac{3 \times N \times 4\ \text{Byte}}{t}
$$

下面的数值来自当前平台，不包含任何硬编码的 8060S、W7900D 或 9070XT 性能值。当前设备的理论峰值未由运行时可靠提供，因此本块只打印实测有效带宽，不猜测利用率。


In [ ]:
actual_vector_size = benchmark_result["vector_size"]
actual_gpu_min_ms = benchmark_result["gpu_min_ms"]
actual_bytes_moved = actual_vector_size * 3 * 4
actual_bandwidth_gb_s = actual_bytes_moved / (actual_gpu_min_ms / 1000) / 1e9

print("当前平台有效带宽")
print(f"- device_name: {benchmark_result['device_name']}")
print(f"- vector_size: {actual_vector_size:,}")
print(f"- logical_bytes_moved: {actual_bytes_moved:,} bytes")
print(f"- gpu_min_ms: {actual_gpu_min_ms:.6f} ms")
print(f"- recomputed_effective_bandwidth: {actual_bandwidth_gb_s:.3f} GB/s")
print(
    "- benchmark_reported_bandwidth: "
    f"{benchmark_result['gpu_bandwidth_gb_s_by_min']:.3f} GB/s"
)
print("- peak_bandwidth_utilization: 未计算（当前设备峰值规格未由本次运行提供）")


#### RX 9070 XT 参考利用率解释

参考记录使用 $N = 16{,}777{,}216$、GPU min 时间约 $0.345\ \text{ms}$，得到约 $583\ \text{GB/s}$ 的逻辑有效带宽。

AMD 官方规格给出的 Radeon RX 9070 XT 显存带宽为 **最高 640 GB/s**，因此参考利用率为：

$$
\text{利用率}_{reference} = \frac{583\ \text{GB/s}}{640\ \text{GB/s}} \approx 91\%
$$

规格来源：[AMD Radeon RX 9070 XT 官方规格](https://www.amd.com/en/support/downloads/drivers.html/graphics/radeon-rx/radeon-rx-9000-series/amd-radeon-rx-9070-xt.html)。这一比例只解释 RX 9070 XT 参考记录；8060S、W7900D 或其他平台不复用该峰值。

对一个如此简单的 kernel 来说，参考平台约 91% 的逻辑有效带宽说明线性、合并访存已经比较有效。剩余差距可能来自 launch 开销、计时边界、缓存与写路径等因素。想进一步提高整体效率，通常需要融合多个逐元素操作，让一次数据搬运完成更多计算，而不是单纯增加算力。

> 完整的 Roofline 读图方法——怎么把工作点画到「算术强度 vs 性能」图上、看它落在带宽斜线还是算力水平线那一侧、据此选排查方向——会在 Part 1 第 7 章系统讲。
>
> **范围说明：** 这里的参考带宽斜线和算力水平线只用于解释 gfx1201 参考环境；当前平台只做平台内趋势判断。

### 4.4.4 何时值得用 GPU

| 场景 | CPU 更合适 | GPU 更合适 |
| ---- | :---: | :---: |
| 数据量小（< 1 万元素） | ✅ | ❌ |
| 单次简单操作、数据不在显存 | ✅ | ❌ |
| 大数据、数据已在显存 | ❌ | ✅ |
| 大数据、多步串联计算 | ❌ | ✅ |
| 计算密集型（如矩阵乘） | ❌ | ✅ |


### Warmup 和 Repeat 的重要性

性能测试中，warmup 和 repeat 是两个基本要素：

- **Warmup**：GPU 初始状态可能不稳定（频率调整、缓存冷启动），先跑几次让它进入稳定工作状态。通常 5-10 次足够。
- **Repeat**：单次测量容易受外部干扰（系统调度、其他进程），多次重复测量可以做统计分析（mean/median/min）。Min 更接近无干扰的真实性能，通常 20-50 次可以得到稳定结果。

下面用代码总结这些最佳实践：

In [ ]:
print("Warmup 的作用:")
print("- GPU 初始状态可能不稳定（频率调整、缓存冷启动）")
print("- Warmup 让 GPU 进入稳定工作状态")
print("- 通常 5-10 次 warmup 足够")

print("\nRepeat 的作用:")
print("- 单次测量容易受外部干扰（系统调度、其他进程）")
print("- 多次重复测量可以统计分析（mean/median/min）")
print("- Min 更接近无干扰的真实性能")
print("- 通常 20-50 次 repeat 可以得到稳定结果")

print("\n最佳实践:")
print("- 正式计时前先 warmup")
print("- 使用 GPU Event 或 synchronize 确保计时准确")
print("- 报告 min/median/mean，不只报告单次结果")
print("- 记录测试参数（size, warmup, repeat）以便复现")

## 4.5 留下实验底稿

跑完前面三段命令，你大概觉得事情已经做完了——其实还没有。**性能工作真正麻烦的一刻，往往不是第一次没跑快，而是过几天回头看时，你自己也说不清当时跑了哪个版本、用了什么输入、那个数字到底是怎么量出来的。** 没留记录的实验，三天后基本等于白做。

所以从这第一个 GPU 程序开始，建议你养成一个小习惯：**每跑完一组实验，顺手把目标、命令、结果记到同一个地方**。在哪里记并不重要——一个 markdown 文件、一份 notebook 都行；重要的是这份记录能在几天后让你（或者别人）一眼看回当初做了什么。

够用的结构其实只有三段：**目标 → 流程 → 结论**。模板示例：

```markdown
# 实验记录：第一个 GPU 程序

## 实验目标
验证 PyTorch ROCm、最小 HIP kernel 和 Vector Add 基准测试是否能跑通。

## 实测流程
cd code/part0-intro
source ./activate-rocm.sh
cd chapter4
hipcc vector_add.hip -O2 -o vector_add
./vector_add
python benchmark_vector_add.py

## 实测结论
| 项目 | 数值 |
| ---- | ---- |
| 硬件 | `<由本次 benchmark 输出填写>` |
| 输入规模 | `<由本次 benchmark 输出填写>` |
| GPU min 延迟 | `<由本次 benchmark 输出填写>` |
| GPU 估算带宽 | `<由本次 benchmark 输出填写>` |
| status | `<由本次 benchmark 输出填写>` |
```

> 下方代码会直接打印一份可复制的本次实验记录，不需要手工抄固定参考值。

> **试一试**：把 `--size` 从默认的 `1 << 24` 改成 `1 << 20` 和 `1 << 26`，分别再跑一次 benchmark，把每次的硬件、输入规模、GPU min 延迟和估算带宽随手记到你的实验记录里。先猜一下——GPU 带宽会一直变大、一直变小，还是先升后降？

In [ ]:
print("## 本次实测结论")
print("| 项目 | 数值 |")
print("| ---- | ---- |")
print(f"| 硬件 | {benchmark_result['device_name']}（{arch}） |")
print(f"| 输入规模 | {benchmark_result['vector_size']:,} 个 float32 |")
print(f"| CPU min 延迟 | {benchmark_result['cpu_min_ms']:.6f} ms |")
print(f"| GPU min 延迟 | {benchmark_result['gpu_min_ms']:.6f} ms |")
print(
    "| GPU 估算有效带宽 | "
    f"{benchmark_result['gpu_bandwidth_gb_s_by_min']:.3f} GB/s |"
)
print(f"| status | {benchmark_result['status']} |")


## 本章小结

- 本章在 `part0-intro` 环境里跑通了第一个手写 HIP Vector Add kernel。
- 第一个 HIP kernel 使用"一线程处理一个元素"的最简单映射方式，方便理解 block、thread 和全局下标。
- 基准测试使用 warmup + repeat，并用 GPU event 计时，避免只量到 CPU 提交开销。
- vector add 的算术强度极低（~0.083 FLOP/Byte），是典型的 memory-bound 算子——性能几乎完全取决于显存带宽，而非算力。
- 当前平台的时间、加速比与有效带宽由 4.4 节动态打印；不在缺少设备峰值规格时猜测利用率。RX 9070 XT 参考记录约为 583/640≈91%，只用于解释参考平台。
- 下一章进入 Part 1 profiling 篇，先系统学怎么量准数字（benchmark 与可信计时）。

## 延伸阅读

- [ROCm HIP Documentation](https://rocm.docs.amd.com/projects/HIP/en/latest/)
- [PyTorch CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html)
- [ROCm System Management Interface](https://rocm.docs.amd.com/projects/rocm_smi_lib/en/latest/)